In [54]:
# Environment Setup
import anndata as ad
import os
import scanpy as sc
from pipeline.utils.env import find_env_dir

ad.settings.allow_write_nullable_strings = True

# Loading .env
h5ad_count_matrix_location = find_env_dir("H5AD_COUNT_MATRIX_DIR")

jakel_series_name = "GSE118257"
jakel = sc.read_h5ad(os.path.join(h5ad_count_matrix_location, jakel_series_name + ".h5ad"))
jakel_oligo = jakel[jakel.obs["celltype"].isin(["OPCs", "COPs", "Oligo1", "Oligo2", "Oligo3", "Oligo4", "Oligo5", "Oligo6", "ImOlGs"])].copy()

absinta_series_name = "GSE180759"
absinta = sc.read_h5ad(os.path.join(h5ad_count_matrix_location, absinta_series_name + ".h5ad"))
absinta_oligo = absinta[absinta.obs["celltype"].isin(["opc", "oligodendrocytes"])].copy()

oligo = ad.concat([jakel_oligo, absinta_oligo], join="inner", label="dataset", fill_value=0)
oligo.write_h5ad(os.path.join(h5ad_count_matrix_location, "oligo.h5ad"))

In [33]:
import numpy as np
import os
import pandas as pd
import scanpy as sc
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot
from pipeline.utils.deseq2 import pseudobulk_deseq2

clustered_data_location = find_env_dir("CLUSTERED_DATA_DIR")
de_analysis_location = find_env_dir("DESEQ_DIR")

series_name = "opc"
oligo = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))
oligo_de_result = pd.read_csv(os.path.join(de_analysis_location, series_name + "_deseq2.csv"), index_col=0)

def category_disease(condition):
    if condition in ["chronic_active_MS_lesion_edge", "A", "CA", "MS_periplaque_white_matter"]:
        return "disease"
    elif condition in ["CI", "chronic_inactive_MS_lesion_edge"]:
        return "devastate"
    else:
        return "control"
    
oligo.obs["disease"] = oligo.obs["condition"].apply(category_disease)

In [3]:
oligo[oligo.obs["leiden"].isin(["1", "3"])]

View of AnnData object with n_obs × n_vars = 2549 × 15381
    obs: 'series', 'batch', 'sample', 'condition', 'celltype', 'dataset', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'leiden', 'disease'
    var: 'mt', 'ribo', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'leiden', 'neighbors', 'umap'
    obsm: 'X_scvi', 'X_umap'
    obsp: 'connectivities', 'distances'

In [68]:
oligo.obs["disease"].value_counts()

disease
disease      33074
devastate    12864
control      10684
Name: count, dtype: Int64

In [ ]:
from pipeline.utils import plot

plot.plot_proportions(oligo, series_name, sample_key="series", group_key="leiden")
plot.plot_proportions(oligo, series_name, sample_key="condition", group_key="leiden")
plot.plot_proportions(oligo, series_name, sample_key="disease", group_key="leiden")
plot.plot_proportions(oligo, series_name, sample_key="sample", group_key="leiden")

config = [{
                "color": "condition",
                "title": "Condition Distribution",
                "legend_loc": "best",
                "palette": None,
        }, {
                "color": "series",
                "title": "Series Distribution",
                "legend_loc": "on data",
                "palette": None,
        }]

plot.plot_umap(oligo, series_name, has_celltype=True, 
               highlight_cells=["OPCs", "COPs", "Oligo1", "Oligo2", "Oligo3", "Oligo4", "Oligo5", "Oligo6", "oligodendrocytes", "opc"], 
               additional_config = config)
plot.plot_proportions(oligo, series_name, sample_key="series", group_key="leiden")

In [9]:
cell_marker_genes = {
    "Newly Formed": ["BCAS1", "ENPP6", "GALC","AARS", "RARS"],
    "TF": [
    "ETV1", "ZEB1", "HIF3A", "SOX5", "SOX6", "BCHE", "FGFR1", "RUNX1T1", 
    "PADI2", "CREM", "ETV6", "ELL2", "ZMIZ1", "BCL6", "KDM6A", "ZEB2", 
    "JUND", "BACH1", "SRPK2", "MLLT3", "CDK8"
],
    "Wish": ["TCP11L2", "PIEZO1", "PIEZO2", "TUBA1A", "TUBA1B", "ACTB", "GAPDH", "PPIA", "PGK1", "RPL13A", "ATF1", "ATF2"],
    "OPC": ["PDGFRA", "OLIG1", "OLIG2", "PCDH15", "MEGF11", "VCAN", ],
    "DEG": ["HAS2", "FERMT1", "CACNG5", "COL9A1", "C1QL2", "SPSB4", "MYT1"],
    "COP": ["BCAN", "ITPR2", "GPR17", "BMP4"],
    "Oligodendrocyte": ["MBP", "PLP1", "MOG", "MAG", "CNP", "MOBP", "CLDN11", "FA2H", 
                        "PPP1R14A", "ERMN", "GPR37", "KLK6", "TF", "UGT8"],
    "Neuron": ["MAP2", "STMN2", "RBFOX3", "SNAP25", "ENO2", "SLC17A7", "SYT1", "CCK", 
               "GABRA1", "GRIN1", "SATB2", "GRIN2B","MYT1L", "SYN1", "TMEM130"],
    "NANI": ["APOE", "CD74"],
    "New Neuron": ["DCX", "GAD1", "GAD2", "CALB2", "RELN"],
    "Astrocyte": ["GFAP", "AQP4", "ALDH1L1", "SOX9", "MLC1", "ATP1B2", "GJA1", "SLC14A1", 
                  "ALDH1A1", "DIO2", "AGT", "ATP13A4", "BMPR1B", "CHI3L1", 
                  "GLIS3", "ID4", "RFX4", "SLC4A4", "SOX2",],
    "Microglia": ["P2RY12", "TMEM119", "CX3CR1", "CSF1R", "TREM2", "ITGAM", "ITGAX", "SLC2A5"],
    "Endothelial": ["CLDN5", "PECAM1", "CD34", "FLT1", "VWF", "CDH5", "CLEC14A", "ERG", "ITM2A", "TM4SF1"],
    "Pericyte": ["PDGFRB", "RGS5", "CD248", "KCNJ8", "NOTCH3"],
    "Ependymal": ["FOXJ1", "DYNLRB2", "PIFO"],
    "Smooth Muscle": ["MYH11", "CNN1", "TAGLN", "ACTA2",],
    "Marker": ["LHFPL3", "LUZP2", "DSCAM", "XYLT1", "EPN2"],
    
    "Lymphocyte": ["CD3E", "CD4", "CD8A", "TRAC"],
    "Macrophage": ["CD68", "MRC1", "LYZ"],
    "VLMC": ["COL1A2",  "DCN"],
}
plot.plot_dotplot(oligo, series_name, cell_marker_genes, group = "leiden")

In [ ]:
oligo.obs["deseq"] = np.where(oligo.obs["leiden"].isin(["1", "3"]), "clust", "other")
oligo_deg = pseudobulk_deseq2(oligo, ["deseq"])
oligo_deg.set_index("gene", inplace=True)
oligo_deg.sort_values("log2FoldChange_shrunk", ascending=False, inplace=True)

In [23]:
oligo[oligo.obs["celltype"] == "COPs"].obs["leiden"].value_counts()

leiden
1    105
0     69
2     62
4      3
3      2
5      1
Name: count, dtype: int64

In [32]:
opc = oligo[oligo.obs["leiden"].isin(["1", "3", "5"])].copy()
series_name = "opc"
import scanpy as sc
import anndata as ad
ad.settings.allow_write_nullable_strings = True
opc.write_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

In [ ]:
oligo[oligo.obs["leiden"] == "3"].obs["sample"].value_counts()

sample
13_047        10960
09_034        10456
12_078         9947
13_015         6250
09_067         5680
14_043         1894
12_002         1530
CO39           1430
11_69          1133
MS122_CI       1117
SD48/16        1074
CO14            970
MS176_NAWM      803
MS122_A         739
MS122_NAWM      500
CO28            368
MS176_RM        361
MS121_CA        328
MS121_NAWM      273
MS121_A3        256
CO25            195
MS121_A2        189
MS176_CA        107
MS176_CI         32
MS242_CA2        24
MS242_CA5         4
MS242_RM          2
Name: count, dtype: int64

In [ ]:
gene = [
    "AFF1", "AFF4", "AGO1", "AHR", "AHRR", "AMH", "APOBEC3B", "AR", "ARID1A", 
    "ARID1B", "ARID2", "ARID3A", "ARID4B", "ARRB1", "ASCL1", "ASCL2", "ASH1L", 
    "ASH2L", "ATF1", "ATF3", "ATF5", "ATF7", "BACH1", "BAP1", "BARX1", "BATF", 
    "BCHE", "BCL11A", "BCL11B", "BCL3", "BCL6", "BCLAF1", "BCOR", "BDP1", 
    "BHLHE22", "BHLHE40", "BMI1", "BRCA1", "BRD2", "BRD3", "BRD4", "BRD7", 
    "BRD9", "BRIP1", "CBX3", "CC2D1A", "CCNT2", "CDC5L", "CDK7", "CDK8", 
    "CDK9", "CDKN1B", "CDX2", "CEBPA", "CEBPB", "CEBPD", "CEBPG", "CHD1", 
    "CHD2", "CHD7", "CHD8", "CLOCK", "CNOT3", "CREB1", "CREBBP", "CREM", 
    "CRY1", "CTBP1", "CTCF", "CTCFL", "CXXC1", "DAND5", "DAXX", "DEK", 
    "DMAP1", "DNMT3A", "DPF2", "DRAP1", "E2F1", "E2F2", "E2F4", "E2F5", 
    "E2F6", "E2F7", "E2F8", "EBF1", "EBF3", "EED", "EHF", "EHMT2", "EGR1", 
    "EGR2", "EGR3", "ELF1", "ELF2", "ELF3", "ELF4", "ELL2", "EOMES", "EP300", 
    "EP400", "EPC1", "EPAS1", "ERCC6", "ERF", "ERG", "ESR1", "ESRRA", "ETS1", 
    "ETV1", "ETV4", "ETV5", "ETV6", "EZH2", "FEZF1", "FGFR1", "FLI1", "FOS", 
    "FOSB", "FOSL1", "FOSL2", "FOXA1", "FOXA2", "FOXA3", "FOXJ2", "FOXK1", 
    "FOXK2", "FOXM1", "FOXO1", "FOXO3", "FOXP1", "FOXP2", "FOXP3", "FXR2", 
    "GABPA", "GABPB1", "GATA1", "GATA2", "GATA3", "GATA4", "GATA6", "GATAD2A", 
    "GATAD2B", "GFI1B", "GLIS1", "GLIS2", "GLIS3", "GMEB1", "GMEB2", "GRHL2", 
    "GRHL3", "GTF2B", "GTF2F1", "GTF3C2", "GTF3C5", "GUCY1B1", "H2AZ1", "HBP1", 
    "HCFC1", "HDAC1", "HDAC2", "HDAC3", "HDAC6", "HDAC8", "HDGFL3", "HES1", 
    "HEY1", "HEYL", "HHEX", "HIC1", "HIF1A", "HIF3A", "HIRA", "HMBOX1", 
    "HMG20A", "HMGN3", "HMGXB4", "HNF1B", "HNF4A", "HNRNPH1", "HNRNPL", "HOXA5", 
    "HOXA6", "HOXB13", "HOXC5", "HSF1", "ID3", "IKZF1", "IKZF2", "IKZF3", 
    "INTS11", "INTS13", "INTS3", "IRF1", "IRF2", "IRF4", "IRF9", "IRX2", 
    "JMJD6", "JUN", "JUNB", "JUND", "KAT7", "KDM1A", "KDM2B", "KDM3A", "KDM4A", 
    "KDM4C", "KDM5B", "KDM5C", "KDM6A", "KDM6B", "KLF1", "KLF4", "KLF5", 
    "KLF6", "KLF8", "KLF9", "KLF10", "KLF11", "KLF13", "KLF15", "KLF16", 
    "KLF17", "KMT2A", "KMT2B", "KMT2C", "KCNH2", "LDB1", "LEF1", "LEO1", 
    "LMNA", "LMO1", "LMO2", "LYL1", "MAF", "MAFF", "MAFK", "MAX", "MAZ", 
    "MBD2", "MBD3", "MBD4", "MBL2", "MCM7", "ME1", "ME3", "MECOM", "MED1", 
    "MED12", "MEF2A", "MEF2B", "MEIS2", "MEN1", "METTL3", "MIER1", "MIER3", 
    "MITF", "MLLT1", "MLLT3", "MNT", "MTOR", "MTA2", "MTA3", "MXI1", "MYB", 
    "MYC", "MYH11", "MYNN", "MYCN", "MYOD1", "NAB2", "NANOG", "NBN", "NCOA1", 
    "NCOA3", "NCOR1", "NCOR2", "NCAPH2", "NELFE", "NEUROD1", "NEUROG2", "NFE2", 
    "NFE2L1", "NFE2L2", "NFIA", "NFIC", "NFKB2", "NFKBIZ", "NFYC", "NFYA", 
    "NFATC1", "NFATC3", "NFAT5", "NIPBL", "NKX2-1", "NKX2-2", "NKX3-1", "NONO", 
    "NOTCH1", "NR1H2", "NR1H3", "NR2C2", "NR2F1", "NR2F2", "NR2F6", "NR3C1", 
    "NR4A1", "NR5A2", "NRF1", "NUP98", "OCA2", "OGG1", "ONECUT2", "ORC1", 
    "ORC2", "OSR2", "OTX2", "OVOL2", "p65", "PADI2", "PARP1", "PATZ1", 
    "PAX5", "PBX4", "PCBP1", "PDX1", "PEX2", "PGR", "PHB2", "PHF5A", "PHF8", 
    "PIAS1", "PKNOX1", "PMEPA1", "PML", "POU2F2", "POU5F1", "PPARG", "PRDM6", 
    "PRDM10", "PRDM14", "PRMT1", "PTBP1", "PTEN", "PYGO2", "RAD21", "RAD51", 
    "RAG2", "RARA", "RARG", "RB1", "RBBP5", "RBPJ", "RCOR1", "RCOR2", "RELA", 
    "REL", "RELB", "REST", "RFX1", "RFX2", "RFX3", "RFXANK", "RING1", "RNF2", 
    "RUNX1", "RUNX1T1", "RUNX2", "RUNX3", "RXRA", "SAFB", "SAP130", "SAP30", 
    "SCML2", "SETDB1", "SIN3A", "SIX5", "SKI", "SKIL", "SMAD1", "SMAD2", 
    "SMAD3", "SMAD4", "SMARCA4", "SMARCB1", "SMARCC1", "SMARCC2", "SMARCE1", 
    "SMC1A", "SMC3", "SNAI2", "SOX2", "SOX4", "SOX5", "SOX6", "SOX9", "SOX13", 
    "SP1", "SP2", "SP3", "SP4", "SP5", "SP7", "SP140", "SPDEF", "SPI1", 
    "SPIB", "SREBF1", "SREBF2", "SRF", "SRPK2", "SS18", "SS18/SSX1 fusion", 
    "SSRP1", "SSU72", "STAG1", "STAT1", "STAT2", "STAT3", "STAT4", "STAT5A", 
    "STAT5B", "STAT6", "SUMO1", "SUMO2", "SUPT5H", "SUZ12", "TAF1", "TAF15", 
    "TAL1", "TARDBP", "TBP", "TBL1XR1", "TBX21", "TBXT", "TCF3", "TCF4", 
    "TCF7", "TCF7L1", "TCF7L2", "TCF12", "TEAD1", "TEAD2", "TEAD4", "TERF1", 
    "TFAP2A", "TFAP2C", "TFAP4", "TFE3", "TGIF2", "TLE3", "TP53", "TP63", 
    "TRIM22", "TRIM24", "TRIM28", "TRPS1", "UBP1", "UBTF", "USF1", "USF2", 
    "USP7", "VDR", "VEZF1", "WRNIP1", "WT1", "XBP1", "XRCC5", "YAP1", "YBX1", 
    "YY1", "ZBED1", "ZBTB5", "ZBTB7A", "ZBTB10", "ZBTB11", "ZBTB14", "ZBTB17", 
    "ZBTB20", "ZBTB26", "ZBTB33", "ZBTB40", "ZBTB42", "ZEB1", "ZEB2", "ZFAT", 
    "ZFP36", "ZFP42", "ZFP91", "ZFX", "ZHX1", "ZHX2", "ZIC2", "ZKSCAN1", 
    "ZMIZ1", "ZNF3", "ZNF18", "ZNF24", "ZNF35", "ZNF76", "ZNF83", "ZNF84", 
    "ZNF143", "ZNF148", "ZNF175", "ZNF184", "ZNF217", "ZNF263", "ZNF264", 
    "ZNF274", "ZNF280A", "ZNF280D", "ZNF281", "ZNF318", "ZNF324", "ZNF366", 
    "ZNF395", "ZNF423", "ZNF467", "ZNF554", "ZNF574", "ZNF577", "ZNF580", 
    "ZNF589", "ZNF629", "ZNF639", "ZNF644", "ZNF692", "ZNF770", "ZNF843", 
    "ZSCAN16", "ZSCAN22", "ZSCAN29"
]
temp = subset[subset.index.isin(gene)]
temp

21

In [5]:
LOG_FOLD_CHANGE_THRESHOLD = 1
ADJUSTED_PVALUE_THRESHOLD = 0.05
BASE_MEAN_THRESHOLD = 100

subset = oligo_deg[(oligo_deg["log2FoldChange_shrunk"] > LOG_FOLD_CHANGE_THRESHOLD) & 
                   (oligo_deg["padj"] < ADJUSTED_PVALUE_THRESHOLD) & 
                   (oligo_deg["baseMean"] > BASE_MEAN_THRESHOLD)]
subset

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,log2FoldChange_shrunk,lfcSE_shrunk,cluster_id,contrast,group_keys
gene,,,,,,,,,,,
MYT1,152.372754,7.548992,0.497840,15.163480,6.171129e-52,5.215282e-50,7.461295,0.585501,clust,rest,deseq
BEST3,109.878468,7.483125,0.468613,15.968675,2.112246e-57,2.620037e-55,7.412383,0.517551,clust,rest,deseq
NXPH1,699.359944,7.468160,0.486560,15.348890,3.602975e-53,3.338395e-51,7.375136,0.601496,clust,rest,deseq
KCND2,887.586327,7.377093,0.452334,16.308943,8.525496e-60,1.234111e-57,7.301377,0.537230,clust,rest,deseq
CA10,966.621976,7.281471,0.446184,16.319429,7.180395e-60,1.051825e-57,7.209675,0.531153,clust,rest,deseq
...,...,...,...,...,...,...,...,...,...,...,...
NENF,181.355296,1.024808,0.174908,5.859131,4.652963e-09,2.528877e-08,1.004905,0.176456,other,rest,deseq
ZNF827,144.544050,1.021771,0.169983,6.011006,1.843752e-09,1.033106e-08,1.003000,0.171810,clust,rest,deseq
SLC38A9,118.227549,1.016856,0.148974,6.825731,8.747868e-12,5.901358e-11,1.002433,0.149752,other,rest,deseq


In [75]:
oligo[oligo.obs["leiden"] == "1"].obs["sample"].value_counts()

sample
13_047        422
09_034        319
13_015        311
09_067        234
12_078        179
SD48/16       174
14_043        127
12_002        105
CO39           59
11_69          56
CO14           47
MS122_CI       35
MS122_NAWM     35
MS122_A        20
MS176_NAWM      9
MS121_CA        7
MS121_A3        6
MS121_NAWM      6
CO28            5
MS176_CA        4
CO25            3
MS176_RM        3
MS242_CA2       3
MS121_A2        2
MS242_CA5       1
MS242_RM        1
Name: count, dtype: int64

In [ ]:
comp = oligo[oligo.obs["leiden"].isin(["1", "3"])]
comp_deg = pseudobulk_deseq2(comp, ["leiden"])

<class 'anndata._core.anndata.AnnData'>
Not enough samples for group  1
Not enough samples for group  3


ValueError: No objects to concatenate